# GDP Prediction — ML (Polynomial, XGBoost) & DL (PyTorch LSTM + Transformer)

Updated notebook: includes train/test RMSE reporting for DL and improved Year labels (e.g. `2024_25`) in plots.

**Notes:** Upload `final GDP Dataset - Copy.csv` if prompted.

In [ ]:
# Install required packages (run in Colab; if packages already installed this is fast)
!pip install -q xgboost joblib

# Imports and device setup
import os, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

## Metrics utilities
We use MAE, RMSE, and R². RMSE reported for train and test where applicable.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)
def rmse(y_true, y_pred):
    return float(math.sqrt(mean_squared_error(y_true, y_pred)))
def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)
def evaluate_metrics(y_true, y_pred):
    return {"MAE": mae(y_true,y_pred), "RMSE": rmse(y_true,y_pred), "R2": r2(y_true,y_pred)}

## Load Data & visualize GDP over years
We preserve the original Year label (e.g. `2024-25`) as `Year_label` and also extract a numeric year for ordering. The plot uses `Year_label` with cleaned formatting like `2024_25` and shows fewer x-ticks for readability.

In [ ]:
from pathlib import Path
expected_path = '/mnt/data/final GDP Dataset - Copy.csv'
local_path = 'final GDP Dataset - Copy.csv'

csv_path = None
if Path(expected_path).exists():
    csv_path = expected_path
elif Path(local_path).exists():
    csv_path = local_path
else:
    try:
        from google.colab import files
        print("Please upload the CSV file (choose 'final GDP Dataset - Copy.csv') — a file dialog will open.")
        uploaded = files.upload()
        csv_path = next(iter(uploaded.keys()))
    except Exception as e:
        raise RuntimeError("CSV not found. Please upload `final GDP Dataset - Copy.csv` to the Colab environment.") from e

print('Using CSV:', csv_path)
df = pd.read_csv(csv_path)

# Preserve original label and create cleaned label for plotting (e.g. '2024-25' -> '2024_25')
df['Year_label'] = df['Year'].astype(str)
# Clean: replace hyphen with underscore, strip spaces
df['Year_label'] = df['Year_label'].str.replace('-', '_').str.replace('/', '_').str.strip()

# Extract numeric year for ordering: prefer first 4-digit group, else use index
df['Year_num'] = df['Year_label'].str.extract(r'(\d{4})')
df['Year_num'] = pd.to_numeric(df['Year_num'], errors='coerce')

# If multiple rows have same Year_num or NaN, fall back to index order
if df['Year_num'].isna().any():
    df['Year_num'] = range(len(df))

target_col = 'Gross Domestic Product__Constant Prices'
if target_col not in df.columns:
    raise RuntimeError(f"Target column '{target_col}' not found. Columns: {df.columns.tolist()}")

display(df.head())

# Plot improved Year vs GDP: use Year_label but reduce ticks
plt.figure(figsize=(14,4))
plt.plot(df['Year_label'], df[target_col], marker='o', linewidth=1)
plt.title('Gross Domestic Product__Constant Prices over Years')
plt.xlabel('Year')
plt.ylabel('Gross Domestic Product__Constant Prices')
plt.grid(alpha=0.3)

# rotate labels and show every Nth tick to avoid clutter
n_ticks = 10
labels = df['Year_label'].tolist()
indices = list(range(0, len(labels), max(1, len(labels)//n_ticks)))
plt.xticks([i for i in indices], [labels[i] for i in indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## ML part: train/test 80/20 (time ordered)
Standard scaling, Polynomial Regression, XGBoost.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['Year','Year_label','Year_num', target_col]]
print('Features:', feature_cols)
X = df[feature_cols].values
y = df[target_col].values

n = len(df)
split_idx = int(n * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
years_ml = df['Year_label'].iloc[split_idx:].values
print(f'Train size: {len(y_train)}, Test size: {len(y_test)}')

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)
X_test_scaled = scaler_ml.transform(X_test)
joblib.dump(scaler_ml, 'scaler_ml.joblib')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipeline = Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)),
                          ('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0, random_state=SEED))])
poly_pipeline.fit(X_train, y_train)
poly_pred = poly_pipeline.predict(X_test)
poly_train_pred = poly_pipeline.predict(X_train)
metrics_poly_test = evaluate_metrics(y_test, poly_pred)
metrics_poly_train = evaluate_metrics(y_train, poly_train_pred)
print('Polynomial TEST metrics:', metrics_poly_test)
print('Polynomial TRAIN metrics:', metrics_poly_train)
joblib.dump(poly_pipeline, 'polynomial_pipeline.joblib')

In [ ]:
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED,
                             n_estimators=200, learning_rate=0.05, max_depth=4, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_train_pred = xgb_model.predict(X_train_scaled)
metrics_xgb_test = evaluate_metrics(y_test, xgb_pred)
metrics_xgb_train = evaluate_metrics(y_train, xgb_train_pred)
print('XGBoost TEST metrics:', metrics_xgb_test)
print('XGBoost TRAIN metrics:', metrics_xgb_train)
joblib.dump(xgb_model, 'xgboost_model.joblib')

## DL prep: feature scaler and target-only scaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler
FEATURE_SCALER = MinMaxScaler()
TARGET_SCALER = MinMaxScaler()

features_all = df[feature_cols].values.astype(np.float32)
target_all = df[target_col].values.reshape(-1,1).astype(np.float32)

features_scaled_all = FEATURE_SCALER.fit_transform(features_all)
target_scaled_all = TARGET_SCALER.fit_transform(target_all)

joblib.dump(FEATURE_SCALER, 'scaler_features_dl.joblib')
joblib.dump(TARGET_SCALER, 'scaler_target_dl.joblib')

### Create sequences (sliding window)

In [ ]:
LOOKBACK = 5
X_seq = []; y_seq = []
for i in range(LOOKBACK, len(df)):
    X_seq.append(features_scaled_all[i-LOOKBACK:i, :])
    y_seq.append(target_scaled_all[i,0])
X_seq = np.array(X_seq); y_seq = np.array(y_seq)
n_seq = X_seq.shape[0]; split_seq = int(n_seq*0.8)
X_train_dl, X_test_dl = X_seq[:split_seq], X_seq[split_seq:]
y_train_dl, y_test_dl = y_seq[:split_seq], y_seq[split_seq:]
years_dl = df['Year_label'].iloc[LOOKBACK + split_seq : LOOKBACK + split_seq + len(y_test_dl)].values
print('DL shapes:', X_train_dl.shape, X_test_dl.shape)

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 8
train_loader = DataLoader(SeqDataset(X_train_dl, y_train_dl), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(SeqDataset(X_test_dl, y_test_dl), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout if num_layers>1 else 0.0)
        self.fc1 = nn.Linear(hidden_size*2, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32,1)
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        out,_ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(self.relu(self.fc1(out)))
        out = self.fc2(out)
        return out

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.input_fc = nn.Linear(input_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(d_model,64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64,1))
    def forward(self,x):
        x = self.input_fc(x)
        x = self.pos_enc(x)
        x = self.transformer_encoder(x)
        x = x.permute(0,2,1)
        x = self.pool(x).squeeze(2)
        out = self.fc(x)
        return out

In [ ]:
def compute_mse_on_loader(model, loader, device=DEVICE):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0.0; total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            total_loss += loss.item() * xb.size(0); total += xb.size(0)
    return total_loss / total if total>0 else 0.0

def train_model_torch(model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr); criterion = nn.MSELoss()
    best_loss = float('inf'); best_state = None; epochs_no_improve = 0
    for epoch in range(1, epochs+1):
        model.train(); running_loss=0.0; total=0
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.to(device)
            opt.zero_grad(); preds = model(xb); loss = criterion(preds, yb); loss.backward(); opt.step()
            running_loss += loss.item()*xb.size(0); total += xb.size(0)
        train_loss = running_loss/total if total>0 else 0.0
        test_loss = compute_mse_on_loader(model, test_loader, device)
        print(f"Epoch {epoch}/{epochs} Train loss: {train_loss:.6f} | Test loss: {test_loss:.6f}")
        if test_loss < best_loss - 1e-8:
            best_loss = test_loss; best_state = {k:v.cpu() for k,v in model.state_dict().items()}; epochs_no_improve=0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            if best_state is not None: model.load_state_dict(best_state)
            print(f"Early stopping at epoch {epoch}"); break
    if best_state is not None: model.load_state_dict(best_state)
    return model

In [ ]:
# Prepare loaders (already defined earlier)
train_loader = DataLoader(SeqDataset(X_train_dl, y_train_dl), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(SeqDataset(X_test_dl, y_test_dl), batch_size=BATCH_SIZE, shuffle=False)

# LSTM
input_size = X_train_dl.shape[2]
lstm_model = LSTMModel(input_size=input_size, hidden_size=64, num_layers=1, dropout=0.2)
print('Training LSTM...')
lstm_model = train_model_torch(lstm_model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE)
torch.save(lstm_model.state_dict(), 'pytorch_lstm.pt')
print('Saved pytorch_lstm.pt')

# Transformer
transformer_model = TransformerModel(input_size=input_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2)
print('Training Transformer...')
transformer_model = train_model_torch(transformer_model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE)
torch.save(transformer_model.state_dict(), 'pytorch_transformer.pt')
print('Saved pytorch_transformer.pt')

## 6) Predictions & Evaluation (DL)
We report TRAIN RMSE and TEST RMSE for the DL models, plus MAE and R² on the DL test set.

In [ ]:
def predict_torch(model, X, device=DEVICE, batch_size=64):
    model.eval()
    ds = SeqDataset(X, np.zeros((len(X),)))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for xb,_ in loader:
            xb = xb.to(device)
            out = model(xb).cpu().numpy().reshape(-1)
            preds.append(out)
    return np.concatenate(preds, axis=0)

# LSTM: train preds scaled -> inverse, test preds scaled -> inverse
y_pred_lstm_train_scaled = predict_torch(lstm_model, X_train_dl)
y_pred_lstm_test_scaled = predict_torch(lstm_model, X_test_dl)
y_pred_lstm_train = TARGET_SCALER.inverse_transform(y_pred_lstm_train_scaled.reshape(-1,1)).reshape(-1)
y_pred_lstm_test = TARGET_SCALER.inverse_transform(y_pred_lstm_test_scaled.reshape(-1,1)).reshape(-1)
y_train_dl_unscaled = TARGET_SCALER.inverse_transform(y_train_dl.reshape(-1,1)).reshape(-1)
y_test_dl_unscaled = TARGET_SCALER.inverse_transform(y_test_dl.reshape(-1,1)).reshape(-1)

# Metrics LSTM
metrics_lstm_train = { 'RMSE_train': rmse(y_train_dl_unscaled, y_pred_lstm_train)}
metrics_lstm_test = evaluate_metrics(y_test_dl_unscaled, y_pred_lstm_test)
print('LSTM TRAIN RMSE:', metrics_lstm_train['RMSE_train'])
print('LSTM TEST metrics:', metrics_lstm_test)

# Transformer
y_pred_trans_train_scaled = predict_torch(transformer_model, X_train_dl)
y_pred_trans_test_scaled = predict_torch(transformer_model, X_test_dl)
y_pred_trans_train = TARGET_SCALER.inverse_transform(y_pred_trans_train_scaled.reshape(-1,1)).reshape(-1)
y_pred_trans_test = TARGET_SCALER.inverse_transform(y_pred_trans_test_scaled.reshape(-1,1)).reshape(-1)

metrics_trans_train = {'RMSE_train': rmse(y_train_dl_unscaled, y_pred_trans_train)}
metrics_trans_test = evaluate_metrics(y_test_dl_unscaled, y_pred_trans_test)
print('\nTransformer TRAIN RMSE:', metrics_trans_train['RMSE_train'])
print('Transformer TEST metrics:', metrics_trans_test)

## 7) Fair comparison (same calendar years)
We compute for each model: RMSE on its training set (RMSE_train) and RMSE on the *same* calendar years (RMSE_test) so comparison has both train/test RMSE.

In [ ]:
# Intersection years as before
years_ml_list = list(years_ml)
years_dl_list = list(years_dl)
intersect_indices_in_ml = [i for i,yv in enumerate(years_ml_list) if yv in years_dl_list]
intersect_years = [years_ml_list[i] for i in intersect_indices_in_ml]

# ML: train RMSE on ML training set, test RMSE on intersection years
poly_train_rmse = rmse(y_train, poly_train_pred)
xgb_train_rmse = rmse(y_train, xgb_train_pred)

ml_actual_on_intersect = y_test[intersect_indices_in_ml]
ml_poly_on_intersect = poly_pred[intersect_indices_in_ml]
ml_xgb_on_intersect = xgb_pred[intersect_indices_in_ml]

poly_test_rmse_on_intersect = rmse(ml_actual_on_intersect, ml_poly_on_intersect)
xgb_test_rmse_on_intersect = rmse(ml_actual_on_intersect, ml_xgb_on_intersect)

# DL: train RMSE already computed above, test RMSE on intersection years (aligned)
dl_year_to_idx = {y: idx for idx,y in enumerate(years_dl_list)}
dl_indices_aligned = [dl_year_to_idx[y] for y in intersect_years]
dl_lstm_on_intersect = y_pred_lstm_test[dl_indices_aligned]  # careful: y_pred_lstm_test aligned already by test order
dl_trans_on_intersect = y_pred_trans_test[dl_indices_aligned]

# DL train RMSEs
lstm_train_rmse = rmse(y_train_dl_unscaled, y_pred_lstm_train)
trans_train_rmse = rmse(y_train_dl_unscaled, y_pred_trans_train)

# DL test RMSEs on intersection (note: subset of test)
lstm_test_rmse_on_intersect = rmse(ml_actual_on_intersect, dl_lstm_on_intersect)
trans_test_rmse_on_intersect = rmse(ml_actual_on_intersect, dl_trans_on_intersect)

comparison_rows = [
    {'Model':'Polynomial Regression', 'RMSE_train': poly_train_rmse, 'RMSE_test': poly_test_rmse_on_intersect, **evaluate_metrics(ml_actual_on_intersect, ml_poly_on_intersect)},
    {'Model':'XGBoost', 'RMSE_train': xgb_train_rmse, 'RMSE_test': xgb_test_rmse_on_intersect, **evaluate_metrics(ml_actual_on_intersect, ml_xgb_on_intersect)},
    {'Model':'LSTM (PyTorch)', 'RMSE_train': lstm_train_rmse, 'RMSE_test': lstm_test_rmse_on_intersect, **evaluate_metrics(ml_actual_on_intersect, dl_lstm_on_intersect)},
    {'Model':'Transformer (PyTorch)', 'RMSE_train': trans_train_rmse, 'RMSE_test': trans_test_rmse_on_intersect, **evaluate_metrics(ml_actual_on_intersect, dl_trans_on_intersect)},
]

comparison_df = pd.DataFrame(comparison_rows)
print('Years compared:', intersect_years)
display(comparison_df)
comparison_df.to_csv('comparison_with_train_test_rmse.csv', index=False)

In [ ]:
# list saved files
print('Saved files (if training completed):')
for f in ['polynomial_pipeline.joblib','xgboost_model.joblib','pytorch_lstm.pt','pytorch_transformer.pt',
          'scaler_ml.joblib','scaler_features_dl.joblib','scaler_target_dl.joblib','comparison_with_train_test_rmse.csv']:
    print('-', f, os.path.exists(f))